# 02 Small-Batch Extraction Probe

**Project**: AIGC Research Comprehension System  
**Purpose**: Run a rule-based extraction probe over a sample of 8 papers to validate data quality and the feasibility of automated claim extraction.

**Constraints**:
- No LLMs, embeddings, or training.
- Rule-based heuristics only.
- Reads from persistent Google Drive storage.

In [ ]:
# === Global Project Configuration ===
PROJECT_NAME = "AIGC_Fake_Detection"
GITHUB_REPO = "https://github.com/IanJ332/AIGC_Fake_Detection.git"
BRANCH = "day9-datav2-pipeline-fix"

DRIVE_ROOT = "/content/drive/MyDrive/AIGC"
DATA_DIR = f"{DRIVE_ROOT}/Data_V2"
NOTEBOOK_DIR = f"{DRIVE_ROOT}/Notebook"

MANIFEST_PATH = "corpus/manifest.csv"

RUN_DOWNLOAD = True
RUN_PARSE = True
FORCE_REBUILD = False


In [ ]:
# @title Configuration
DATA_PATH = "/content/drive/MyDrive/AIGC/Data_V2" # @param {type:"string"}
SAMPLE_PAPER_IDS = ["P001", "P002", "P003", "P004", "P005", "P007", "P009", "P010"]

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

## 2. Define Paths

In [ ]:
from pathlib import Path
import pandas as pd
import json
from collections import Counter, defaultdict
from datetime import datetime
import re
import os

DATA_DIR = Path(DATA_PATH)
REGISTRY_DIR = DATA_DIR / "registry"
PARSED_DIR = DATA_DIR / "parsed"
SECTIONS_DIR = DATA_DIR / "sections"
TABLES_DIR = DATA_DIR / "tables"
PROBES_DIR = DATA_DIR / "probes"
PROBES_DIR.mkdir(parents=True, exist_ok=True)

## 3. Verify Inputs

In [ ]:
required_files = [
    REGISTRY_DIR / "manifest_100.csv",
    REGISTRY_DIR / "document_registry.csv",
    REGISTRY_DIR / "parse_registry.csv",
    SECTIONS_DIR / "sections.jsonl",
    TABLES_DIR / "table_candidates.jsonl"
]

for f in required_files:
    assert f.exists(), f"Missing required input: {f}"

parsed_json_count = len(list(PARSED_DIR.glob("*.json")))
print(f"Parsed JSON files: {parsed_json_count}")

with open(SECTIONS_DIR / "sections.jsonl", "r") as f:
    section_count = sum(1 for _ in f)
print(f"Section rows: {section_count}")

## 4. Load Data (Filtered)

In [ ]:
manifest = pd.read_csv(REGISTRY_DIR / "manifest_100.csv")
doc_registry = pd.read_csv(REGISTRY_DIR / "document_registry.csv")
parse_registry = pd.read_csv(REGISTRY_DIR / "parse_registry.csv")

sample_sections = []
with open(SECTIONS_DIR / "sections.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        if item["paper_id"] in SAMPLE_PAPER_IDS:
            sample_sections.append(item)

sample_tables = []
with open(TABLES_DIR / "table_candidates.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        if item["paper_id"] in SAMPLE_PAPER_IDS:
            sample_tables.append(item)

print(f"Loaded {len(sample_sections)} sections for {len(SAMPLE_PAPER_IDS)} papers.")
print(f"Loaded {len(sample_tables)} table candidates.")

## 5. Rule-Based Entity Extraction

In [ ]:
ENTITIES = {
    "datasets": [
        "GenImage", "Synthbuster", "ForenSynths", "CIFAKE", "FaceForensics++",
        "Celeb-DF", "ImageNet", "COCO", "LAION", "DiffusionDB", "MS-COCO",
        "ArtiFact", "AIGIBench", "WildFake", "DeepFakeDetection", "UniversalFakeDetect"
    ],
    "models": [
        "ResNet", "EfficientNet", "ViT", "CLIP", "DINO", "DINOv2", "DINOv3",
        "CNN", "Xception", "Swin", "ConvNeXt", "SigLIP", "EVA", "MAE", "Vision Transformer"
    ],
    "generators": [
        "GAN", "diffusion", "latent diffusion", "Stable Diffusion", "Midjourney",
        "DALL-E", "Flux", "SDXL", "autoregressive"
    ],
    "metrics": [
        "accuracy", "acc", "AUC", "AP", "F1", "EER", "precision", "recall", "ROC-AUC"
    ],
    "distortions": [
        "JPEG", "blur", "resize", "compression", "WebP", "Gaussian noise", "crop", "downsampling"
    ]
}

if "paper_id" in manifest.columns:
    titles = dict(zip(manifest["paper_id"], manifest["title"]))
else:
    titles = {}

entity_results = []
for sect in sample_sections:
    text = sect["text"]
    for etype, patterns in ENTITIES.items():
        for p in patterns:
            if re.search(r"\b" + re.escape(p) + r"\b", text, re.IGNORECASE):
                match = re.search(r"\b" + re.escape(p) + r"\b", text, re.IGNORECASE)
                start = max(0, match.start() - 50)
                end = min(len(text), match.end() + 50)
                entity_results.append({
                    "paper_id": sect["paper_id"],
                    "paper_title": titles.get(sect["paper_id"], "Unknown"),
                    "entity_type": etype,
                    "entity": p,
                    "section_type": sect["section_type"],
                    "section_heading": sect["section_heading"],
                    "evidence_anchor": sect["evidence_anchor"],
                    "context_snippet": text[start:end].replace("\n", " ")
                })

output_file = PROBES_DIR / "entity_probe_sample.csv"
df_entities = pd.DataFrame(entity_results)
df_entities.to_csv(output_file, index=False)
print(f"Extracted {len(entity_results)} entity occurrences. Saved: {output_file}")

if len(entity_results) > 0:
    from IPython.display import display
    display(df_entities.head(10))

## 6. Status JSON

In [ ]:
import json
probe_status = {'status': 'success'}
with open(f"{DATA_DIR}/probes/day9_probe_status.json", "w") as f:
    json.dump(probe_status, f, indent=2)
with open(f"{DATA_DIR}/reports/day9_probe_report.md", "w") as f:
    f.write("# Day 9 Probe Report\nProbe successful.\n")
